## Step 5 - Build the FAISS index (+ id_map)

Stacks the per-frame CLIP vectors into one FAISS inner-product index and writes
the **aligned id_map** the backend needs. Without `id_map.json` the backend
cannot turn an index row back into a `frame_id`.

```
clip.index       # FAISS IndexFlatIP  -> data/indexes/<model>.index
id_map.json      # ["L21_V001_000080", ...] aligned to index rows -> data/indexes/id_map.json
```

The `frame_id` (`<video_name>_<idx>`) matches the keys in `frames_meta.json`.

In [ ]:
!pip -q install faiss-cpu

In [ ]:
import os
import glob
import json
import faiss
import numpy as np
from pathlib import Path
from tqdm import tqdm

In [ ]:
FEATURES_DIR = '/kaggle/input/aic-clip-features/clip-features'   # output of Step 4
INDEX_PATH   = './clip.index'
ID_MAP_PATH  = './id_map.json'

In [ ]:
npy_paths = sorted(glob.glob(os.path.join(FEATURES_DIR, '**', '*.npy'), recursive=True))
print('vectors:', len(npy_paths))
assert npy_paths, 'No .npy features found - check FEATURES_DIR'

ids, vectors = [], []
for p in tqdm(npy_paths):
    path = Path(p)
    video_name = path.parent.name          # L21_V001 (thư mục cha trực tiếp của file)
    idx = int(path.stem)                    # 000080 -> 80
    ids.append(f'{video_name}_{idx:06d}')   # frame_id, matches frames_meta.json
    vectors.append(np.load(p).astype(np.float32).reshape(-1))

matrix = np.vstack(vectors)
faiss.normalize_L2(matrix)                  # guarantee unit norm -> IP == cosine
dim = matrix.shape[1]
print('matrix:', matrix.shape)

In [ ]:
index = faiss.IndexFlatIP(dim)
index.add(matrix)
faiss.write_index(index, INDEX_PATH)
with open(ID_MAP_PATH, 'w', encoding='utf-8') as f:
    json.dump(ids, f, ensure_ascii=False)

print('index vectors:', index.ntotal)
print('wrote', INDEX_PATH, 'and', ID_MAP_PATH)

### Deploy

Copy the three artifacts into the backend:

```
clip.index       -> backend/data/indexes/clip.index
id_map.json      -> backend/data/indexes/id_map.json
frames_meta.json -> backend/data/metadata/frames_meta.json   (from Step 2)
Keyframes/*      -> backend/media/frames/                    (from Step 3)
```

Query the API with `model=clip` (the index file is `clip.index`).